# Big Data / Ανάλυση Δεδομένων Μεγάλου Όγκου  
## 2η Υποχρεωτική Ομαδική Εργασία (2025)

**Ονοματεπώνυμοι & Αριθμοί Μητρώου:**  
- Κωνσταντίνος Μιχαηλίδης (ics23148)  
- Τασσόπουλος Παντελής (ics22002)

**Περιβάλλον Υλοποίησης:**  
Google Colab (Linux x86_64, Python 3.12.12, Apache Spark / PySpark 4.0.1)

**Ημερομηνία:**  
13/12/2025

In [2]:
import pyspark
pyspark.__version__

'4.0.1'

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Telecom Churn Analysis") \
    .getOrCreate()

In [6]:
df = spark.read.csv(
    "/content/drive/MyDrive/telecom_churn_10k.csv",
    header=True,
    inferSchema=True
)

In [7]:
df.show(10)

+-----------+------+----+-------+-------------+--------------+------------+----------+------+---------------+-------------+--------------+-------------+--------------+-----+
|CUSTOMER_ID|GENDER| AGE|COUNTRY|TENURE_MONTHS| CONTRACT_TYPE|HAS_INTERNET|HAS_MOBILE|HAS_TV|MONTHLY_CHARGES|TOTAL_CHARGES|NUM_COMPLAINTS|SUPPORT_CALLS|PAYMENT_METHOD|CHURN|
+-----------+------+----+-------+-------------+--------------+------------+----------+------+---------------+-------------+--------------+-------------+--------------+-----+
|    C000001|  Male|68.0|     FR|         61.0|Month-to-month|           0|         1|     0|          17.54|      1077.63|             0|            0| Bank transfer|  1.0|
|    C000002|Female|65.0|     FR|         NULL|Month-to-month|           1|         1|     0|          50.82|       833.94|             1|            1|   Credit card|  0.0|
|    C000003|  Male|36.0|     FR|         53.0|      Two year|           1|         1|     0|          34.44|       1828.5|       

In [8]:
df.printSchema()

root
 |-- CUSTOMER_ID: string (nullable = true)
 |-- GENDER: string (nullable = true)
 |-- AGE: double (nullable = true)
 |-- COUNTRY: string (nullable = true)
 |-- TENURE_MONTHS: double (nullable = true)
 |-- CONTRACT_TYPE: string (nullable = true)
 |-- HAS_INTERNET: integer (nullable = true)
 |-- HAS_MOBILE: integer (nullable = true)
 |-- HAS_TV: integer (nullable = true)
 |-- MONTHLY_CHARGES: double (nullable = true)
 |-- TOTAL_CHARGES: double (nullable = true)
 |-- NUM_COMPLAINTS: integer (nullable = true)
 |-- SUPPORT_CALLS: integer (nullable = true)
 |-- PAYMENT_METHOD: string (nullable = true)
 |-- CHURN: double (nullable = true)



In [9]:
df.count()

10000

In [11]:
from pyspark.sql.functions import col, isnan, when, count, lit
from pyspark.sql.types import DoubleType, FloatType, IntegerType, LongType, ShortType, DecimalType

missing_summary = None

for c in df.columns:
    # Get the data type of the current column
    col_type = df.schema[c].dataType

    # Start with the condition to check for NULL values
    missing_condition = col(c).isNull()

    # If the column is of a numeric type, also check for NaN values
    if isinstance(col_type, (DoubleType, FloatType, IntegerType, LongType, ShortType, DecimalType)):
        missing_condition = missing_condition | isnan(col(c))

    temp = df.select(
        lit(c).alias("column_name"),
        count(when(missing_condition, c)).alias("missing_count")
    )
    missing_summary = temp if missing_summary is None else missing_summary.union(temp)

missing_summary.show(truncate=False)

+---------------+-------------+
|column_name    |missing_count|
+---------------+-------------+
|CUSTOMER_ID    |0            |
|GENDER         |0            |
|AGE            |300          |
|COUNTRY        |0            |
|TENURE_MONTHS  |300          |
|CONTRACT_TYPE  |0            |
|HAS_INTERNET   |0            |
|HAS_MOBILE     |0            |
|HAS_TV         |0            |
|MONTHLY_CHARGES|300          |
|TOTAL_CHARGES  |300          |
|NUM_COMPLAINTS |0            |
|SUPPORT_CALLS  |0            |
|PAYMENT_METHOD |0            |
|CHURN          |300          |
+---------------+-------------+



### Έλεγχος ελλιπών τιμών

Από τον πίνακα ελλιπών τιμών παρατηρείται ότι missing values εμφανίζονται κυρίως
στα πεδία AGE, TENURE_MONTHS, MONTHLY_CHARGES, TOTAL_CHARGES και CHURN.
Ιδιαίτερα κρίσιμες για την ανάλυση είναι οι ελλιπείς τιμές στο CHURN, καθώς πρόκειται
για τη μεταβλητή–στόχο, καθώς και στα MONTHLY_CHARGES και TENURE_MONTHS,
που επηρεάζουν άμεσα τόσο την ανάλυση churn όσο και τα επόμενα μοντέλα.

In [13]:
df_clean = df.filter(col("CHURN").isNotNull())

In [14]:
numeric_cols = ["AGE", "TENURE_MONTHS", "MONTHLY_CHARGES", "TOTAL_CHARGES"]

for c in numeric_cols:
    mean_value = df_clean.select(c).na.drop().agg({c: "mean"}).first()[0]
    df_clean = df_clean.fillna({c: mean_value})

### Στρατηγική καθαρισμού δεδομένων

Οι γραμμές με ελλιπείς τιμές στο πεδίο CHURN αφαιρέθηκαν, καθώς το CHURN αποτελεί
τη μεταβλητή–στόχο και η παρουσία missing τιμών θα δημιουργούσε ασάφεια στην ανάλυση.
Για τα αριθμητικά πεδία AGE, TENURE_MONTHS, MONTHLY_CHARGES και TOTAL_CHARGES
επιλέχθηκε η αντικατάσταση των ελλιπών τιμών με τον μέσο όρο, ως μια απλή και
τεκμηριωμένη προσέγγιση που διατηρεί το μέγεθος του dataset.

In [15]:
df_clean.select(
    "AGE", "TENURE_MONTHS", "MONTHLY_CHARGES", "TOTAL_CHARGES"
).describe().show()

+-------+------------------+------------------+------------------+------------------+
|summary|               AGE|     TENURE_MONTHS|   MONTHLY_CHARGES|     TOTAL_CHARGES|
+-------+------------------+------------------+------------------+------------------+
|  count|              9700|              9700|              9700|              9700|
|   mean| 51.42650653629523|36.559855411439315|36.556964285714315|1339.5033949633366|
| stddev|19.247481813671897|20.466005655823807|11.127497787046334| 875.0902702308767|
|    min|              18.0|               1.0|               5.0|               0.0|
|    max|              85.0|              72.0|             80.45|           5197.62|
+-------+------------------+------------------+------------------+------------------+



In [16]:
df_clean.groupBy("CHURN").agg(
    {"TENURE_MONTHS": "mean", "MONTHLY_CHARGES": "mean"}
).show()

+-----+--------------------+------------------+
|CHURN|avg(MONTHLY_CHARGES)|avg(TENURE_MONTHS)|
+-----+--------------------+------------------+
|  0.0|   36.27477349916856|38.328541395976266|
|  1.0|   37.03490974729247| 33.56423868839328|
+-----+--------------------+------------------+



### Περιγραφική ανάλυση

Από τα αποτελέσματα παρατηρείται ότι οι πελάτες που έχουν αποχωρήσει (CHURN = 1)
εμφανίζουν κατά μέσο όρο μικρότερη διάρκεια συνεργασίας σε σχέση με τους ενεργούς
πελάτες. Επιπλέον, οι αποχωρήσαντες πελάτες τείνουν να έχουν υψηλότερες μηνιαίες
χρεώσεις, γεγονός που υποδηλώνει ότι τόσο η παλαιότητα όσο και η τιμολόγηση
σχετίζονται με τη συμπεριφορά churn.

In [17]:
from pyspark.sql.functions import when

df_clean = df_clean.withColumn(
    "NUM_SERVICES",
    col("HAS_INTERNET") + col("HAS_MOBILE") + col("HAS_TV")
)

df_clean = df_clean.withColumn(
    "AVG_CHARGE_PER_MONTH",
    when(col("TENURE_MONTHS") > 0,
         col("TOTAL_CHARGES") / col("TENURE_MONTHS")
    ).otherwise(0)
)

df_clean = df_clean.withColumn(
    "IS_LONG_TENURE",
    when(col("TENURE_MONTHS") >= 24, 1).otherwise(0)
)

In [18]:
df_clean.select(
    "AGE", "TENURE_MONTHS", "NUM_SERVICES",
    "AVG_CHARGE_PER_MONTH", "IS_LONG_TENURE"
).show(10)

+-----------------+-----------------+------------+--------------------+--------------+
|              AGE|    TENURE_MONTHS|NUM_SERVICES|AVG_CHARGE_PER_MONTH|IS_LONG_TENURE|
+-----------------+-----------------+------------+--------------------+--------------+
|             68.0|             61.0|           1|  17.666065573770492|             1|
|             65.0|36.55985541143951|           2|   22.81026526541081|             1|
|             36.0|             53.0|           2|                34.5|             1|
|             23.0|              6.0|           3|               45.26|             0|
|             44.0|             39.0|           3|  38.953589743589745|             1|
|             70.0|             25.0|           3|             57.7488|             1|
|             65.0|             42.0|           2|   47.51690476190476|             1|
|51.42650653629504|             42.0|           2|  29.615476190476187|             1|
|             76.0|             59.0|      

In [19]:
df_clean.select(
    [count(when(col(c).isNull() | isnan(col(c)), c)).alias(c)
     for c in ["AGE","TENURE_MONTHS","MONTHLY_CHARGES","TOTAL_CHARGES","CHURN"]]
).show()

+---+-------------+---------------+-------------+-----+
|AGE|TENURE_MONTHS|MONTHLY_CHARGES|TOTAL_CHARGES|CHURN|
+---+-------------+---------------+-------------+-----+
|  0|            0|              0|            0|    0|
+---+-------------+---------------+-------------+-----+



Ολοκλήρωση Θέματος 1

## Θέμα 2: Ανάλυση με Spark SQL & Επιβεβαίωση Επιχειρησιακών Μοτίβων

In [20]:
df_clean.createOrReplaceTempView("churn_view")

In [21]:
q1 = spark.sql("""
SELECT
  CONTRACT_TYPE,
  COUNT(*) AS total_customers,
  SUM(CASE WHEN CHURN = 1 THEN 1 ELSE 0 END) AS churn_customers,
  ROUND(100.0 * SUM(CASE WHEN CHURN = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_pct
FROM churn_view
GROUP BY CONTRACT_TYPE
ORDER BY churn_pct DESC
""")

q1.show(truncate=False)

+--------------+---------------+---------------+---------+
|CONTRACT_TYPE |total_customers|churn_customers|churn_pct|
+--------------+---------------+---------------+---------+
|Month-to-month|5326           |2834           |53.21    |
|One year      |2440           |503            |20.61    |
|Two year      |1934           |264            |13.65    |
+--------------+---------------+---------------+---------+



In [22]:
q2 = spark.sql("""
SELECT
  CASE
    WHEN NUM_SERVICES >= 3 THEN '3+'
    WHEN NUM_SERVICES = 2 THEN '2'
    WHEN NUM_SERVICES = 1 THEN '1'
    ELSE '0'
  END AS services_group,
  COUNT(*) AS total_customers,
  SUM(CASE WHEN CHURN = 1 THEN 1 ELSE 0 END) AS churn_customers,
  ROUND(100.0 * SUM(CASE WHEN CHURN = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_pct
FROM churn_view
GROUP BY
  CASE
    WHEN NUM_SERVICES >= 3 THEN '3+'
    WHEN NUM_SERVICES = 2 THEN '2'
    WHEN NUM_SERVICES = 1 THEN '1'
    ELSE '0'
  END
ORDER BY
  CASE services_group
    WHEN '0' THEN 0
    WHEN '1' THEN 1
    WHEN '2' THEN 2
    WHEN '3+' THEN 3
  END
""")

q2.show(truncate=False)

+--------------+---------------+---------------+---------+
|services_group|total_customers|churn_customers|churn_pct|
+--------------+---------------+---------------+---------+
|0             |229            |99             |43.23    |
|1             |2149           |1031           |47.98    |
|2             |5152           |1926           |37.38    |
|3+            |2170           |545            |25.12    |
+--------------+---------------+---------------+---------+



In [23]:
q3 = spark.sql("""
SELECT
  CHURN,
  ROUND(AVG(MONTHLY_CHARGES), 2) AS avg_monthly_charges
FROM churn_view
GROUP BY CHURN
ORDER BY CHURN
""")

q3.show(truncate=False)

+-----+-------------------+
|CHURN|avg_monthly_charges|
+-----+-------------------+
|0.0  |36.27              |
|1.0  |37.03              |
+-----+-------------------+



In [24]:
q3b = spark.sql("""
SELECT
  CONTRACT_TYPE,
  CHURN,
  ROUND(AVG(MONTHLY_CHARGES), 2) AS avg_monthly_charges
FROM churn_view
GROUP BY CONTRACT_TYPE, CHURN
ORDER BY CONTRACT_TYPE, CHURN
""")

q3b.show(truncate=False)

+--------------+-----+-------------------+
|CONTRACT_TYPE |CHURN|avg_monthly_charges|
+--------------+-----+-------------------+
|Month-to-month|0.0  |41.23              |
|Month-to-month|1.0  |38.5               |
|One year      |0.0  |35.26              |
|One year      |1.0  |33.47              |
|Two year      |0.0  |30.05              |
|Two year      |1.0  |28.15              |
+--------------+-----+-------------------+



In [25]:
q4 = spark.sql("""
SELECT
  COUNTRY,
  COUNT(*) AS total_customers,
  SUM(CASE WHEN CHURN = 1 THEN 1 ELSE 0 END) AS churn_customers,
  ROUND(100.0 * SUM(CASE WHEN CHURN = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_pct
FROM churn_view
GROUP BY COUNTRY
HAVING COUNT(*) >= 100
ORDER BY churn_pct DESC
LIMIT 5
""")

q4.show(truncate=False)

+-------+---------------+---------------+---------+
|COUNTRY|total_customers|churn_customers|churn_pct|
+-------+---------------+---------------+---------+
|IT     |1003           |409            |40.78    |
|DE     |1198           |451            |37.65    |
|GR     |3732           |1382           |37.03    |
|UK     |1560           |564            |36.15    |
|ES     |1050           |379            |36.10    |
+-------+---------------+---------------+---------+



### Έλεγχος επιχειρησιακών hints

Από το ερώτημα churn ανά τύπο συμβολαίου (q1), το Month-to-month εμφανίζει churn **53.21%**,
ενώ το One year **20.61%** και το Two year **13.65%**. Αυτό δείχνει ξεκάθαρα ότι το Month-to-month έχει
πολύ υψηλότερο churn σε σχέση με τα συμβόλαια μεγαλύτερης διάρκειας, άρα το πρώτο hint **επιβεβαιώνεται**.
Από το churn ανά πλήθος υπηρεσιών (q2), οι πελάτες με 1 υπηρεσία έχουν churn **47.98%**,
με 2 υπηρεσίες **37.38%**, ενώ με 3+ υπηρεσίες **25.12%**. Παρατηρούμε ότι όσο αυξάνεται το πλήθος υπηρεσιών,
το ποσοστό churn μειώνεται σημαντικά, άρα το hint ότι «πολλαπλές υπηρεσίες (NUM_SERVICES ≥ 3) → χαμηλότερο churn»
**επιβεβαιώνεται** στα δεδομένα.
Τέλος, από το q3 η μέση MONTHLY_CHARGES για CHURN=0 είναι **36.27€**, ενώ για CHURN=1 είναι **37.03€**.
Οι churners έχουν ελαφρώς υψηλότερη μέση μηνιαία χρέωση (διαφορά ~0.76€), άρα το τρίτο hint
«υψηλότερες MONTHLY_CHARGES → αυξημένο churn» **επιβεβαιώνεται με μικρή ένταση**.
Συνολικά, τα αποτελέσματα δείχνουν ότι ο τύπος συμβολαίου και το πλήθος υπηρεσιών επηρεάζουν πολύ έντονα το churn,
ενώ η μηνιαία χρέωση φαίνεται να έχει πιο ήπια συσχέτιση.

Ολοκλήρωση Θέματος 2

## Θέμα 3: Πρόβλεψη MONTHLY_CHARGES με Decision Tree (Spark ML Regression)

In [26]:
# Label
label_col = "MONTHLY_CHARGES"

# Features (όπως ζητά η εκφώνηση)
numeric_features = [
    "AGE", "TENURE_MONTHS", "NUM_COMPLAINTS", "SUPPORT_CALLS",
    "HAS_INTERNET", "HAS_MOBILE", "HAS_TV", "NUM_SERVICES"
]

categorical_features = ["CONTRACT_TYPE", "PAYMENT_METHOD", "COUNTRY"]

selected_cols = numeric_features + categorical_features + [label_col]

model_df = df_clean.select(*selected_cols)

# Έλεγχος ότι δεν υπάρχουν missing στα πεδία που θα χρησιμοποιηθούν
model_df = model_df.na.drop(subset=selected_cols)

model_df.show(5, truncate=False)
model_df.printSchema()

+----+-----------------+--------------+-------------+------------+----------+------+------------+--------------+--------------+-------+---------------+
|AGE |TENURE_MONTHS    |NUM_COMPLAINTS|SUPPORT_CALLS|HAS_INTERNET|HAS_MOBILE|HAS_TV|NUM_SERVICES|CONTRACT_TYPE |PAYMENT_METHOD|COUNTRY|MONTHLY_CHARGES|
+----+-----------------+--------------+-------------+------------+----------+------+------------+--------------+--------------+-------+---------------+
|68.0|61.0             |0             |0            |0           |1         |0     |1           |Month-to-month|Bank transfer |FR     |17.54          |
|65.0|36.55985541143951|1             |1            |1           |1         |0     |2           |Month-to-month|Credit card   |FR     |50.82          |
|36.0|53.0             |0             |0            |1           |1         |0     |2           |Two year      |Credit card   |FR     |34.44          |
|23.0|6.0              |1             |1            |1           |1         |1     |3   

In [27]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor

# StringIndexers για κατηγορικά
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_features
]

# OneHotEncoder (προαιρετικό στην εκφώνηση - το βάζουμε για καλύτερη συμπεριφορά στα κατηγορικά)
encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in categorical_features],
    outputCols=[f"{c}_ohe" for c in categorical_features],
    handleInvalid="keep"
)

# Τελική λίστα features για assembler
assembler_inputs = numeric_features + [f"{c}_ohe" for c in categorical_features]

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features",
    handleInvalid="keep"
)

regressor = DecisionTreeRegressor(
    labelCol=label_col,
    featuresCol="features",
    seed=42
)

pipeline = Pipeline(stages=indexers + [encoder, assembler, regressor])

In [28]:
train_df, test_df = model_df.randomSplit([0.7, 0.3], seed=42)

model = pipeline.fit(train_df)

predictions = model.transform(test_df)

predictions.select(label_col, "prediction").show(10, truncate=False)

+------------------+------------------+
|MONTHLY_CHARGES   |prediction        |
+------------------+------------------+
|36.44             |39.696780424983885|
|41.9              |39.696780424983885|
|36.556964285714415|30.168208802212483|
|40.71             |39.696780424983885|
|44.76             |35.420169521455065|
|18.04             |35.420169521455065|
|36.73             |39.696780424983885|
|55.55             |46.98294248949576 |
|39.74             |39.696780424983885|
|40.73             |51.617908866995066|
+------------------+------------------+
only showing top 10 rows


In [29]:
from pyspark.ml.evaluation import RegressionEvaluator

rmse_evaluator = RegressionEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="rmse"
)

r2_evaluator = RegressionEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="r2"
)

rmse = rmse_evaluator.evaluate(predictions)
r2 = r2_evaluator.evaluate(predictions)

print(f"RMSE: {rmse:.4f}")
print(f"R^2:  {r2:.4f}")

RMSE: 7.8793
R^2:  0.4945


In [30]:
from pyspark.sql.functions import avg

avg_charge = model_df.select(avg(label_col)).first()[0]
print(f"Μέση τιμή MONTHLY_CHARGES στο dataset: {avg_charge:.2f} €")

Μέση τιμή MONTHLY_CHARGES στο dataset: 36.56 €


### Αξιολόγηση μοντέλου (RMSE & R²)

Το μοντέλο Decision Tree παρήγαγε RMSE = **7.8793** και R² = **0.4945** στο test set.
Δεδομένου ότι η μέση μηνιαία χρέωση στο dataset είναι περίπου **36.56€**, ένα RMSE της τάξης των **7.88€**
σημαίνει ότι η πρόβλεψη τείνει να αποκλίνει κατά ~8€ από την πραγματική τιμή, δηλαδή περίπου **20%** της μέσης χρέωσης.
Το R² ≈ **0.49** δείχνει ότι το μοντέλο εξηγεί περίπου το μισό της διακύμανσης των MONTHLY_CHARGES, άρα είναι
χονδρικά χρήσιμο για απλές “what-if” προσεγγίσεις, αλλά τα σφάλματα δεν είναι αρκετά μικρά για ακριβή τιμολόγηση.

Ολοκλήρωση Θέματος 3